# Langfuse Local Smoke Test

Use this notebook to verify that Python can send traces to your local Langfuse instance.

Before running it:

1. Start Langfuse with Docker Compose.
2. Open `http://localhost:3000`.
3. Create a project and API keys.
4. Put the keys in a local `.env` file or in your shell environment.

Expected environment variables for Langfuse:

```bash
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_BASE_URL=http://localhost:3000
# Optional compatibility alias for older examples:
LANGFUSE_HOST=http://localhost:3000
```

A Langfuse project is the workspace that owns these API keys. Traces sent with the same project keys appear together in that project's tracing view. For this notebook, keep the no-LLM smoke test, the LangChain/OpenAI test, and the direct OpenAI test in the same project so they are easy to compare.

The LangChain/OpenAI example also needs:

```bash
OPENAI_API_KEY=sk-...
```

## Example 1

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

if not os.getenv("LANGFUSE_BASE_URL") and os.getenv("LANGFUSE_HOST"):
    os.environ["LANGFUSE_BASE_URL"] = os.environ["LANGFUSE_HOST"]

required_env_vars = ["LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL"]

missing = [name for name in required_env_vars if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Missing Langfuse environment variables: {', '.join(missing)}")

print(f"Langfuse base URL: {os.environ['LANGFUSE_BASE_URL']}")

Langfuse base URL: http://localhost:3000


The `@observe()` decorator creates a trace/span for the wrapped function. In notebooks and short scripts, `flush()` is important because the SDK sends events asynchronously.

In [2]:
from langfuse import observe, get_client
 
 
@observe(name="local-langfuse-smoke-test")
def greet(name: str) -> str:
    message = f"Hello, {name}!"
    get_client().update_current_span(
        input={"name": name},
        output={"message": message},
        metadata={
            "module": "04_Monitoring_LangFuse",
            "example": "01_langfuse_test",
            "user_id": "course-student",
            "session_id": "notebook-01",
            "tags": ["local", "notebook", "smoke-test"],
        },
    )
    get_client().set_current_trace_io(input={"name": name}, output={"message": message})
    return message


message = greet("Langfuse")
get_client().flush()
 
print(message)

C:\Users\A200239740\AppData\Local\Temp\ipykernel_1040\3486395481.py:18: DeprecationWarning: Trace-level input/output is deprecated. For trace attributes (user_id, session_id, tags, etc.), use propagate_attributes() instead. This method will be removed in a future major version.
  get_client().set_current_trace_io(input={"name": name}, output={"message": message})


Hello, Langfuse!


Now open the Langfuse UI and go to the project's **Tracing** view. You should see a trace named `local-langfuse-smoke-test`.

## Example 2: Minimal LangChain/OpenAI Trace

This second example sends a real LLM request through LangChain. It uses the same Langfuse project keys, so the trace should appear beside the smoke-test trace.

In [3]:
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Missing OPENAI_API_KEY for the LangChain/OpenAI example")

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langfuse.langchain import CallbackHandler


langfuse_handler = CallbackHandler()

prompt = ChatPromptTemplate.from_template(
    "Answer in one short paragraph: {question}"
)
model = ChatOpenAI(model="gpt-4o-mini")
chain = prompt | model

response = chain.invoke(
    {"question": "Why is observability useful for LLM applications?"},
    config={
        "callbacks": [langfuse_handler],
        "metadata": {"example": "langchain-openai-minimal"},
        "tags": ["local", "notebook", "langchain"],
    },
)

get_client().flush()

print(response.content)

Observability is crucial for LLM applications as it enables developers and operators to monitor and understand the behavior of language models in real time. By providing insights into model performance, response accuracy, and usage patterns, observability helps identify issues, optimize performance, and ensure reliability. It also aids in debugging, fine-tuning, and maintaining ethical standards by allowing for better tracking of biases and unintended outputs, ultimately leading to more effective and responsible AI deployment.


Return to the same Langfuse project. You should now see a second trace from the LangChain/OpenAI call, including the prompt/model observation captured by the callback handler.

## Example 3: Minimal OpenAI SDK Trace Without LangChain

This example calls OpenAI directly through Langfuse's OpenAI wrapper. It uses the same `OPENAI_API_KEY` and the same Langfuse project keys, but there is no LangChain prompt, chain, or callback handler involved.

In [6]:
from langfuse.openai import OpenAI


client = OpenAI()

completion = client.chat.completions.create(
    name="openai-sdk-minimal",
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You explain technical ideas clearly and briefly.",
        },
        {
            "role": "user",
            "content": "In one paragraph, explain why LLM tracing is useful.",
        },
    ],
    metadata={"example": "openai-sdk-minimal"},
)

get_client().flush()

print(completion.choices[0].message.content)

LLM tracing is useful because it enables the systematic tracking and analysis of the decision-making processes of large language models (LLMs) during text generation. By capturing the internal states and pathways that lead to specific outputs, tracing helps researchers and developers understand how LLMs interpret prompts, manage context, and generate responses. This transparency aids in identifying biases, improving model performance, and enhancing trustworthiness, as it allows for better accountability and fine-tuning of the models to align with desired outcomes. Furthermore, LLM tracing can inform debugging, model validation, and the development of safer AI systems.


Compare the Langfuse traces:

- The smoke test shows your own traced Python function.
- The LangChain example shows a LangChain run tree captured by the callback handler.
- The direct OpenAI example shows the OpenAI API request without LangChain in the middle.

Example 1:

![Example 1](../assets/langfuse_quickstart_example_1.png)

Example 2:

![Example 2](../assets/langfuse_quickstart_example_2.png)

Example 3:

![Example 3](../assets/langfuse_quickstart_example_3.png)